# Download product as stream to S3

In this tutorial, we'll use [EOProduct.stream_download()](../../api_reference/eoproduct.rst#eodag.api.product._product.EOProduct.stream_download) method to download its data as a stream and upload it on a S3 bucket without storing it locally.

## Configure and prepare S3

In this tutorial, we'll run a local S3 using [rustfs](https://rustfs.com/) and `Docker`, but feel free to use another one.

In a terminal, execute the following command to run a minimal local S3 using default settings:

```sh
docker run --rm -p 9000:9000 rustfs/rustfs:latest
```

Connect to running S3:

In [1]:
import boto3

# default rustfs settings
S3_HOST = "http://127.0.0.1:9000"
S3_USERNAME = "rustfsadmin"
S3_PASSWORD = "rustfsadmin"

# desired upload settings
S3_BUCKET = "test"
CHUNK_SIZE = 8388608  # 8Mo

# Connect to local S3
s3 = boto3.client(
    "s3",
    aws_access_key_id=S3_USERNAME,
    aws_secret_access_key=S3_PASSWORD,
    endpoint_url=S3_HOST
)

List existing buckets:

In [2]:
buckets = s3.list_buckets()
buckets_names = [b.get('Name') for b in buckets.get('Buckets', [])]
buckets_names

[]

Create desired bucket if it does not exist, then list avaible buckets again:

In [3]:
if S3_BUCKET not in buckets_names:
    s3.create_bucket(ACL='private', Bucket=S3_BUCKET)

buckets = s3.list_buckets()
buckets_names = [b.get('Name') for b in buckets.get('Buckets', [])]
buckets_names

['test']

Check bucket content:

In [4]:
s3.list_objects_v2(Bucket=S3_BUCKET).get("Contents", [])

[]

## Search using EODAG

Make sure you configured [cop_dataspace](../../providers.rst#cop-dataspace) credentials, or use another data provider.

In [5]:
from eodag import EODataAccessGateway

dag = EODataAccessGateway()
results = dag.search(
    provider="cop_dataspace",
    collection="S2_MSI_L1C",
    start="2026-01-01",
    end="2026-01-31",
)
product = results[0]
product

EOProduct(id=S2C_MSIL1C_20260101T000441_N0511_R073_T57NUF_20260101T013646, provider=cop_dataspace)

## Upload to S3

In [6]:
from tqdm.auto import tqdm

stream = product.stream_download()

s3_key = "{}/{}".format(
    product.properties["title"],
    stream.filename
)

with tqdm(unit="B", unit_scale=True) as pbar:
    s3.upload_fileobj(
        stream.content,
        S3_BUCKET,
        s3_key,
        Config=boto3.s3.transfer.TransferConfig(multipart_threshold=CHUNK_SIZE),
        Callback=pbar.update
    )

0.00B [00:00, ?B/s]

Uploaded file should now be listed in bucket content:

In [7]:
s3.list_objects_v2(Bucket=S3_BUCKET).get("Contents", [])

[{'Key': 'S2C_MSIL1C_20260101T000441_N0511_R073_T57NUF_20260101T013646/S2C_MSIL1C_20260101T000441_N0511_R073_T57NUF_20260101T013646.zip',
  'LastModified': datetime.datetime(2026, 2, 20, 17, 9, 42, 997000, tzinfo=tzutc()),
  'ETag': '"183e1c7afa997137f28c7c29093f85a7-3"',
  'Size': 20619376,
  'StorageClass': 'STANDARD'}]

<div class="alert alert-info">

Note

Available assets can also be downloaded as stream using [Asset.stream_download()](../../api_reference/assets.rst#eodag.api.product._assets.Asset.stream_download):

```py
product.assets["foo"].stream_download()
```

</div>